# Phase 1 - Data Validation

Validates the two things Phase 1 depends on before any training starts:
1. `EndoSLAMStomachDataset` actually indexes the real EndoSLAM folder layout (its `_index_sequences()` is currently a *guess* -- see the module docstring).
2. `dark_degradation.py`'s synthetic dark-frame model produces something that looks like real endoscope footage, not just a dimmed photo.

**Run this on Kaggle** (GPU + dataset access; matches `config.yaml`'s `/kaggle/input/endoslam` path). Colab works too if Kaggle quota runs out -- just change the `!git clone` cell's working directory assumptions and mount the dataset from wherever you've put it; nothing else in this notebook is Kaggle-specific.

**Round-trip rule:** any fix you make here to `_index_sequences()` or `_load_poses()` (very likely, since they're unverified) must be copied back into local `src/data/endoslam_dataset.py` and committed -- don't leave the real fix stranded in this notebook.

## 0. Setup

Clones the project repo so `src/` here matches what's tracked in git.

In [ ]:
REPO_URL = "https://github.com/ritiksharma3/endoslam.git"
assert REPO_URL, "Set REPO_URL to the pushed GitHub repo before running on Kaggle"

!git clone $REPO_URL repo
%cd repo
!pip install -q -r environment/requirements.txt  # does NOT include torch -- Kaggle ships it preinstalled

import sys
sys.path.insert(0, ".")

## 1. Inspect the real folder layout

`_index_sequences()` in `src/data/endoslam_dataset.py` assumes:
```
{root}/{organ}/{camera}/{sequence_name}/frames/*.png (or .jpg)
{root}/{organ}/{camera}/{sequence_name}/pose.txt
{root}/{organ}/{camera}/{sequence_name}/depth/*.png   (UnityCam only)
```
This is a guess (the official repo only shows the tree as an image). Walk the real thing first.

In [ ]:
import os

DATA_ROOT = "/kaggle/input/endoslam"

for root, dirs, files in os.walk(DATA_ROOT):
    depth = root[len(DATA_ROOT):].count(os.sep)
    print(root, dirs[:5], files[:5])
    if depth > 4:
        break  # don't flood output

## CHECKPOINT -- patch here before continuing

Compare the walk output above against the assumed layout. If they differ (likely):
1. Edit `_index_sequences()` (and `_load_poses()`'s pose-file column-count assumption) in `src/data/endoslam_dataset.py` **in this cloned `repo/` checkout**.
2. Re-run the smoke test cell below until it passes.
3. Copy the working patch back to the local machine's `src/data/endoslam_dataset.py` and commit it there -- this notebook's copy is disposable, the local repo is the source of truth.

Do not proceed to Phase 2 with an unpatched, unverified loader.

## 2. Dataset loader smoke test

In [ ]:
import yaml

with open("configs/config.yaml") as f:
    config = yaml.safe_load(f)

config["data"]["root"] = DATA_ROOT
config

In [ ]:
from src.data.endoslam_dataset import EndoSLAMStomachDataset

all_cameras = [config["data"]["synthetic_cam"]] + config["data"]["real_cams"]

train_ds = EndoSLAMStomachDataset(
    config, split="train", cameras=all_cameras,
    context_window=config["reconstruction"]["context_window"],
)
val_ds = EndoSLAMStomachDataset(
    config, split="val", cameras=all_cameras,
    context_window=config["reconstruction"]["context_window"],
)
test_ds = EndoSLAMStomachDataset(
    config, split="test", cameras=all_cameras,
    context_window=config["reconstruction"]["context_window"],
)

print(f"train windows: {len(train_ds)}")
print(f"val windows:   {len(val_ds)}")
print(f"test windows:  {len(test_ds)}")
assert len(train_ds) > 0, "empty dataset -- _index_sequences() almost certainly needs patching, see CHECKPOINT above"

In [ ]:
sample = train_ds[0]
for k, v in sample.items():
    shape = tuple(v.shape) if hasattr(v, "shape") else v
    print(f"{k}: {shape}")

assert sample["images"].shape[0] == config["reconstruction"]["context_window"]
print("\ncamera / has_depth check across a few samples (expect True only for UnityCam):")
for i in range(0, len(train_ds), max(1, len(train_ds) // 10)):
    s = train_ds[i]
    print(f"  {s['camera']:>8}  has_depth={bool(s['has_depth'].any())}")

## 3. Dark-degradation visual check on real frames

The module's own `__main__` self-test only checks a synthetic gradient. This checks it on an actual EndoSLAM frame -- it should look like dim, slightly blurred endoscope footage, not just a darkened photo.

In [ ]:
import matplotlib.pyplot as plt
from src.data.dark_degradation import build_paired_dataset_entry

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for col in range(4):
    sample = train_ds[col * max(1, len(train_ds) // 4)]
    clean = sample["images"][0].permute(1, 2, 0).numpy()  # first frame of the window, CHW -> HWC
    pair = build_paired_dataset_entry(clean, config)

    axes[0, col].imshow(pair["clean"])
    axes[0, col].set_title(f"clean ({sample['camera']})")
    axes[0, col].axis("off")
    axes[1, col].imshow(pair["dark"])
    axes[1, col].set_title("degraded")
    axes[1, col].axis("off")
plt.tight_layout()
plt.show()

## Done

If the smoke test passed and the degraded frames look plausibly dark/blurred (not just dimmed), Phase 1 is validated. Copy any `_index_sequences()` / `_load_poses()` patches back to local `src/`, commit, and update `PROGRESS.md` before starting Phase 2.